# Step 7: Custom Tools & API Integration

        _Learner Notebook_  
        Estimated time: **75 minutes**  
        Difficulty: **Intermediate-Advanced**

        ## Learning Objectives
        - [ ] Build async tools for remote APIs.
- [ ] Add retries and defensive error handling.
- [ ] Keep the tool contract readable even when the implementation is async.
- [ ] Use live API results in a model answer.

        ## Prerequisites
        - Step 6 completed.
- Comfort with `aiohttp` and async functions.


## Table of Contents

1. Setup & Imports
2. Part 1: Introduction
3. Part 2: Core Implementation
4. Part 3: Hands-On Exercises
5. Part 4: Solutions & Discussion
6. Part 5: Best Practices & Tips
7. Summary & Key Takeaways
8. Additional Resources


## Setup & Imports

The next cell adds the repository root to `sys.path` and imports the shared course helpers.
Most notebooks use the same bootstrap so the lesson content can stay focused on the concept,
not on path setup.


In [ ]:
from pathlib import Path
import sys

PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / "helpers").exists():
    PROJECT_ROOT = PROJECT_ROOT.parent
sys.path.insert(0, str(PROJECT_ROOT))

from helpers import (
    LocalTfidfVectorStore,
    SQLiteConversationMemory,
    assert_minimum_python,
    chunk_documents,
    create_agent,
    create_chat_client,
    describe_response,
    load_settings,
    load_text_documents,
    print_banner,
    print_json,
    resolve_notebook_root,
    summarize_session,
    validate_provider_configuration,
)


## Part 1: Introduction

Remote APIs introduce latency, partial failures, and payload transformation. This notebook shows how to wrap that complexity in a tool the model can still use safely.

Expected output and validation notes:

Expected output snapshot:

- The tool fetches Wikipedia data successfully
- The model summarizes the returned snippets in a clean answer


## Part 2: Core Implementation

### Async Wikipedia tool


In [ ]:
import aiohttp
from agent_framework import tool

@tool
async def search_wikipedia(query: str) -> str:
    """Search Wikipedia and return up to three short snippets."""
    url = "https://en.wikipedia.org/w/api.php"
    params = {
        "action": "query",
        "list": "search",
        "srsearch": query,
        "format": "json",
    }
    async with aiohttp.ClientSession() as session:
        async with session.get(url, params=params, timeout=10) as response:
            response.raise_for_status()
            payload = await response.json()
    results = payload.get("query", {}).get("search", [])
    if not results:
        return "No Wikipedia results found."
    return "\n".join(
        f"- {item['title']}: {item['snippet'][:120]}..."
        for item in results[:3]
    )

api_agent = create_agent(
    name="WikiAgent",
    instructions="Use the Wikipedia tool for fact-finding questions and summarize the results clearly.",
    tools=[search_wikipedia],
)


## Part 2: Core Implementation

### Live API-backed question


In [ ]:
response = await api_agent.run("Give me a short introduction to Python generators.")
print(response.text)


## Part 3: Hands-On Exercises

Wrap the Wikipedia request in a retry helper so transient network errors do not fail the whole tool immediately.

Try the exercise yourself before looking at the solutions in Part 4.


In [ ]:
# TODO: create a retrying wrapper around the HTTP request
# TODO: call it from a second version of the tool


## Part 4: Solutions & Discussion

Use this section to compare your implementation with one clear working approach. The goal is not
perfect matching output; the goal is a sound pattern that is easy to explain, debug, and extend.


In [ ]:
import asyncio

async def fetch_json_with_retry(url: str, params: dict, retries: int = 3) -> dict:
    for attempt in range(retries):
        try:
            async with aiohttp.ClientSession() as session:
                async with session.get(url, params=params, timeout=10) as response:
                    response.raise_for_status()
                    return await response.json()
        except Exception:
            if attempt == retries - 1:
                raise
            await asyncio.sleep(2 ** attempt)
    raise RuntimeError("Unreachable")

@tool
async def resilient_wikipedia(query: str) -> str:
    """Search Wikipedia with retry handling."""
    payload = await fetch_json_with_retry(
        "https://en.wikipedia.org/w/api.php",
        {
            "action": "query",
            "list": "search",
            "srsearch": query,
            "format": "json",
        },
    )
    results = payload.get("query", {}).get("search", [])
    return "\n".join(item["title"] for item in results[:3]) or "No results"


## Part 5: Best Practices & Tips

        - Keep remote tools narrow and easy to test.
- Transform large JSON payloads before handing them back to the model.
- Plan for rate limits, timeouts, and empty results from the beginning.


## Summary & Key Takeaways

You now have a pattern for async, API-backed tools that still feels clean at the agent layer.


## Additional Resources

        - `aiohttp documentation`
- `Wikipedia API reference`
